# 04 — Final Model and Bundle

This notebook consumes only the persisted Dry Bean preparation and model-selection lineage. It reconstructs the frozen multiclass candidate, fits once on train+validation, opens the sealed test only after the fitted-contract gate, evaluates it once, and materializes a reloadable v2 artifact set. It performs no model selection, retuning, feature ablation, resampling comparison, or Notebook 05 inference demo.

## 1. Finalization Context and Boundary

Macro F1 remains the primary descriptive metric, but no selection occurs here. The seven-class decision rule is argmax over contract-ordered scores or probabilities. Binary positive-class and threshold semantics are not applicable. Operational validity remains unconfirmed.

In [37]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path


def _bootstrap_project_root() -> Path:
    configured = os.getenv("DATASET_STUDY_ROOT")
    candidates = ([Path(configured).expanduser()] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        root = candidate.resolve()
        if (root / "scripts" / "finalize_model.py").is_file() and (root / "pyproject.toml").is_file():
            return root
    raise RuntimeError("Dataset-study project root not found.")


PROJECT_ROOT = _bootstrap_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

from scripts.export_figures import export_figure
from scripts.finalize_model import (
    ArtifactConflictError,
    EvaluationGuard,
    assemble_multiclass_final_training_data,
    describe_multiclass_fitted_pipeline,
    evaluate_multiclass_final_model_once,
    freeze_multiclass_finalization_decisions,
    inspect_final_artifact_set,
    load_multiclass_test_partition_after_fit,
    reconstruct_multiclass_selected_pipeline,
    runtime_versions,
    semantic_fingerprint,
    sha256_file,
    validate_existing_multiclass_finalization_equivalence,
    validate_multiclass_upstream_lineage_metadata_only,
    validate_upstream_handoff_contracts,
    verify_multiclass_pipeline_contract,
    write_multiclass_final_model_artifacts,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (9, 5)
print("Multiclass finalization utilities loaded.")

Multiclass finalization utilities loaded.


## 2. Independent Handoff Lineage Validation

The metadata-only gate validates preparation components, prepared/split byte hashes, the model-selection artifact set, and cross-handoff contracts without parsing the test CSV. This path also supports an equivalent rerun without reopening test.

In [38]:
DATASET_SLUG = "dry-bean"
PREPARATION_HANDOFF_PATH = Path("artifacts/preparation/dry-bean/preparation-handoff.json")
MODEL_SELECTION_HANDOFF_PATH = Path("artifacts/model-selection/dry-bean/model-selection-handoff.json")
FINAL_MODEL_ROOT = Path("artifacts/models/dry-bean")

metadata_manifests, model_selection_handoff = validate_multiclass_upstream_lineage_metadata_only(
    project_root=PROJECT_ROOT,
    preparation_handoff_path=PREPARATION_HANDOFF_PATH,
    model_selection_handoff_path=MODEL_SELECTION_HANDOFF_PATH,
)
feature_manifest = metadata_manifests["feature_manifest"]
split_manifest = metadata_manifests["split_manifest"]
preparation_manifest = metadata_manifests["preparation_manifest"]

TEST_INTEGRITY_REFERENCE = {
    "path": split_manifest["partition_paths"]["test"],
    "row_count": split_manifest["row_counts"]["test"],
    "class_counts": split_manifest["class_counts"]["test"],
    "sha256": split_manifest["partition_sha256"]["test"],
    "sealed": model_selection_handoff["test_partition_sealed"],
    "evaluated": model_selection_handoff["test_partition_evaluated"],
}
print("Metadata-only upstream lineage gate: PASSED")
print(json.dumps(TEST_INTEGRITY_REFERENCE, indent=2, sort_keys=True))

Metadata-only upstream lineage gate: PASSED
{
  "class_counts": {
    "BARBUNYA": 199,
    "BOMBAY": 78,
    "CALI": 244,
    "DERMASON": 532,
    "HOROZ": 289,
    "SEKER": 304,
    "SIRA": 396
  },
  "evaluated": false,
  "path": "data/processed/dry-bean/splits/stratified-70-15-15-seed-42/test.csv",
  "row_count": 2042,
  "sealed": true,
  "sha256": "5e17c1346e67cb4a786ebd11162b7e45099ded342b9de9755ab4f18f4068c175"
}


## 3. Frozen Finalization Contract

The contract below is reconstructed from persisted values. It freezes the selected HGB family, all 16 features including ShapeFactor2, exact hyperparameters, no scaling, no categorical processing, no class weighting, no resampling, the target order, and the one-time test access rule.

In [39]:
FINAL_CONTRACT = freeze_multiclass_finalization_decisions(
    dataset_slug=DATASET_SLUG,
    model_selection_handoff=model_selection_handoff,
    feature_manifest=feature_manifest,
    split_manifest=split_manifest,
    model_selection_handoff_path=MODEL_SELECTION_HANDOFF_PATH,
    model_selection_handoff_sha256=sha256_file(PROJECT_ROOT / MODEL_SELECTION_HANDOFF_PATH),
)
assert FINAL_CONTRACT.decision_rule == "argmax_class_score_or_probability"
assert FINAL_CONTRACT.training_partitions == ("train", "validation")
assert FINAL_CONTRACT.evaluation_partition == "test"
assert "ShapeFactor2" in FINAL_CONTRACT.feature_columns
assert dict(FINAL_CONTRACT.imbalance_policy) == {"class_weight": None, "resampling": "none", "strategy": "none"}

display(pd.DataFrame({
    "feature": FINAL_CONTRACT.feature_columns,
    "technical_order": range(len(FINAL_CONTRACT.feature_columns)),
}))
print(json.dumps(FINAL_CONTRACT.as_dict(), indent=2, sort_keys=True))

,feature,technical_order
0,Area,0
1,Perimeter,1
2,MajorAxisLength,2
3,MinorAxisLength,3
4,AspectRatio,4
5,Eccentricity,5
6,ConvexArea,6
7,EquivDiameter,7
8,Extent,8
9,Solidity,9


{
  "binary_threshold": "not_applicable",
  "categorical_features": [],
  "dataset_slug": "dry-bean",
  "decision_rule": "argmax_class_score_or_probability",
  "evaluation_partition": "test",
  "feature_columns": [
    "Area",
    "Perimeter",
    "MajorAxisLength",
    "MinorAxisLength",
    "AspectRatio",
    "Eccentricity",
    "ConvexArea",
    "EquivDiameter",
    "Extent",
    "Solidity",
    "Roundness",
    "Compactness",
    "ShapeFactor1",
    "ShapeFactor2",
    "ShapeFactor3",
    "ShapeFactor4"
  ],
  "feature_policy": "all_features",
  "hyperparameters": {
    "model__class_weight": null,
    "model__l2_regularization": 0.0,
    "model__learning_rate": 0.05,
    "model__max_iter": 250,
    "model__max_leaf_nodes": 15,
    "model__min_samples_leaf": 40
  },
  "identifier_columns": [],
  "imbalance_policy": {
    "class_weight": null,
    "resampling": "none",
    "strategy": "none"
  },
  "model_family": "HistGradientBoostingClassifier",
  "model_id": "hist_gradient_boosti

## 4. Idempotent Artifact-State Gate

A partial set or a semantically divergent complete set fails closed. A complete equivalent set is validated and reused before any DataFrame test access, fit, or evaluation.

In [40]:
FINAL_STATE = inspect_final_artifact_set(PROJECT_ROOT / FINAL_MODEL_ROOT)
if FINAL_STATE == "partial":
    raise ArtifactConflictError("Partial final artifact set detected; manual diagnosis is required.")

IDEMPOTENT_REUSE = FINAL_STATE == "complete"
if IDEMPOTENT_REUSE:
    validate_existing_multiclass_finalization_equivalence(
        output_directory=PROJECT_ROOT / FINAL_MODEL_ROOT,
        contract=FINAL_CONTRACT,
    )
    print("Complete equivalent v2 artifact set validated; fit and test evaluation will not rerun.")
else:
    print("No final Dry Bean artifact set exists; controlled first finalization will proceed.")

Complete equivalent v2 artifact set validated; fit and test evaluation will not rerun.


## 5. Independent Preparation Loading and Test-Object Disposal

On first materialization, the preparation loader validates the complete persisted handoff. Only train, validation, and manifests are retained. The object that structurally exposes test is immediately discarded; before final fit only test path, row count, class counts, hash, and sealed status remain.

In [41]:
if not IDEMPOTENT_REUSE:
    preparation, reloaded_selection = validate_upstream_handoff_contracts(
        project_root=PROJECT_ROOT,
        preparation_handoff_path=PREPARATION_HANDOFF_PATH,
        model_selection_handoff_path=MODEL_SELECTION_HANDOFF_PATH,
    )
    train_frame = preparation.train
    validation_frame = preparation.validation
    preparation_manifests = preparation.manifests
    assert reloaded_selection == model_selection_handoff
    assert preparation_manifests["split_manifest"]["partition_sha256"]["test"] == TEST_INTEGRITY_REFERENCE["sha256"]
    del preparation, reloaded_selection
    print("Preparation handoff reloaded; test-bearing handoff object discarded.")
else:
    print("Idempotent reuse: preparation DataFrames were not loaded.")

Idempotent reuse: preparation DataFrames were not loaded.


## 6. Final Training Dataset Assembly

Train and validation are concatenated in persisted row order. Test is not an accepted argument to the assembly helper.

In [42]:
if not IDEMPOTENT_REUSE:
    final_training = assemble_multiclass_final_training_data(
        train=train_frame,
        validation=validation_frame,
        contract=FINAL_CONTRACT,
    )
    del train_frame, validation_frame
    assert final_training.row_count == split_manifest["row_counts"]["train"] + split_manifest["row_counts"]["validation"]
    assert list(final_training.features.columns) == list(FINAL_CONTRACT.feature_columns)
    assert set(final_training.target.unique()) == set(FINAL_CONTRACT.target_classes)
    display(pd.DataFrame([{
        "fit_rows": final_training.row_count,
        "feature_count": len(FINAL_CONTRACT.feature_columns),
        **{f"class_{name}": count for name, count in final_training.class_counts},
    }]))
else:
    print("Idempotent reuse: final training data assembly skipped.")

Idempotent reuse: final training data assembly skipped.


## 7. Frozen Pipeline Reconstruction

The helper reconstructs the exact `sklearn.pipeline.Pipeline` used for the selected candidate: numerical passthrough followed by `HistGradientBoostingClassifier` with handoff-supplied parameters and estimator seed.

In [43]:
if not IDEMPOTENT_REUSE:
    final_pipeline = reconstruct_multiclass_selected_pipeline(contract=FINAL_CONTRACT)
    verify_multiclass_pipeline_contract(
        final_pipeline, contract=FINAL_CONTRACT, require_fitted=False
    )
    print(final_pipeline)
    print("Frozen model parameters:", dict(FINAL_CONTRACT.hyperparameters))
else:
    print("Idempotent reuse: pipeline reconstruction skipped.")

Idempotent reuse: pipeline reconstruction skipped.


## 8. Single Final Train + Validation Fit

This is the only final fit. There is no cross-validation, search, alternative candidate, feature ablation, or imbalance-policy comparison.

In [44]:
if not IDEMPOTENT_REUSE:
    fit_started = time.perf_counter()
    final_pipeline.fit(final_training.features, final_training.target)
    FIT_DURATION_SECONDS = time.perf_counter() - fit_started
    verify_multiclass_pipeline_contract(
        final_pipeline, contract=FINAL_CONTRACT, require_fitted=True
    )
    print({
        "fit_rows": final_training.row_count,
        "class_counts": dict(final_training.class_counts),
        "fit_duration_seconds": FIT_DURATION_SECONDS,
        "runtime_versions": runtime_versions(),
    })
else:
    print("Idempotent reuse: final fit skipped.")

Idempotent reuse: final fit skipped.


## 9. Fitted Model Contract Verification

The fitted estimator class set must equal the target contract set. Its internal class order is recorded separately from the output class order and never assumed to match it.

In [45]:
if not IDEMPOTENT_REUSE:
    fitted_descriptor = describe_multiclass_fitted_pipeline(
        pipeline=final_pipeline,
        contract=FINAL_CONTRACT,
    )
    FITTED_STATE_FINGERPRINT = semantic_fingerprint(fitted_descriptor)
    assert set(fitted_descriptor["estimator_class_order"]) == set(FINAL_CONTRACT.target_classes)
    assert fitted_descriptor["output_class_order"] == list(FINAL_CONTRACT.target_classes)
    print("Estimator class order:", fitted_descriptor["estimator_class_order"])
    print("Output class order:", fitted_descriptor["output_class_order"])
    print("Fitted-state fingerprint:", FITTED_STATE_FINGERPRINT)
else:
    print("Idempotent reuse: fitted-contract verification will be performed by fresh-process reload.")

Idempotent reuse: fitted-contract verification will be performed by fresh-process reload.


## 10. Sealed Test Access Gate

Only after the frozen contract, final training assembly, completed fit, and fitted-pipeline verification may the helper verify the test SHA-256 and parse the CSV.

In [46]:
if not IDEMPOTENT_REUSE:
    test_partition = load_multiclass_test_partition_after_fit(
        project_root=PROJECT_ROOT,
        fitted_pipeline=final_pipeline,
        contract=FINAL_CONTRACT,
    )
    assert test_partition.row_count == TEST_INTEGRITY_REFERENCE["row_count"]
    assert test_partition.partition_sha256 == TEST_INTEGRITY_REFERENCE["sha256"]
    print("Sealed test access gate: PASSED after final fit")
else:
    print("Idempotent reuse: test was not reopened.")

Idempotent reuse: test was not reopened.


## 11. Single Final Multiclass Test Evaluation

One explicit guard permits one probability call. Probabilities are validated, reordered from estimator order to target-contract order, and used for the official aggregate, per-class, log-loss, and fixed-order confusion evidence.

In [47]:
if not IDEMPOTENT_REUSE:
    evaluation_guard = EvaluationGuard()
    final_evaluation = evaluate_multiclass_final_model_once(
        fitted_pipeline=final_pipeline,
        test_partition=test_partition,
        final_training_features=final_training.features,
        contract=FINAL_CONTRACT,
        validation_evidence=model_selection_handoff["selected_validation_evidence"],
        guard=evaluation_guard,
    )
    assert evaluation_guard.probability_call_count == 1
    assert final_evaluation.test_probability_evaluation_count == 1
    assert final_evaluation.output_class_order == FINAL_CONTRACT.target_classes
    display(pd.DataFrame([final_evaluation.metrics]))
    display(pd.DataFrame(final_evaluation.per_class))
else:
    print("Idempotent reuse: final test evaluation skipped.")

Idempotent reuse: final test evaluation skipped.


## 12. Per-Class, Confusion, and Generalization Review

Validation-to-test deltas, worst-recall classes, ranked confusion pairs, and repeated-profile sensitivity are descriptive only. They cannot change the frozen model or policies.

In [48]:
if not IDEMPOTENT_REUSE:
    generalization_table = pd.DataFrame(
        final_evaluation.validation_to_test["per_class_recall"]
    )
    confusion_pairs = pd.DataFrame(
        final_evaluation.confusion_pair_comparison["ranked_test_pairs"]
    )
    display(pd.DataFrame([final_evaluation.validation_to_test["aggregate_deltas"]]))
    display(generalization_table)
    display(confusion_pairs.head(10))
    print(json.dumps(final_evaluation.repeated_profile_sensitivity, indent=2, sort_keys=True))
else:
    existing_evidence = json.loads(
        (PROJECT_ROOT / FINAL_MODEL_ROOT / "final-test-evidence.json").read_text(encoding="utf-8")
    )
    print("Existing official test metrics:", existing_evidence["metrics"])
    print("Existing test evaluation count:", existing_evidence["test_partition_evaluation_count"])
    del existing_evidence

Existing official test metrics: {'accuracy': 0.9324191968658179, 'balanced_accuracy': 0.9398971908347276, 'log_loss': 0.1810239003137349, 'macro_f1': 0.9418353636024855, 'macro_recall': 0.9398971908347276, 'minimum_per_class_recall': 0.8686868686868687, 'row_count': 2042, 'weighted_f1': 0.9321874638643965}
Existing test evaluation count: 1


## 13. Compact Visual Diagnostics

Only the fixed-order test confusion matrix, validation-versus-test aggregate metrics, and per-class recall comparison are shown.

In [49]:
if not IDEMPOTENT_REUSE:
    matrix = np.asarray(final_evaluation.confusion_matrix["counts"], dtype=int)
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(FINAL_CONTRACT.target_classes)), FINAL_CONTRACT.target_classes, rotation=45, ha="right")
    ax.set_yticks(range(len(FINAL_CONTRACT.target_classes)), FINAL_CONTRACT.target_classes)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title("Final test confusion matrix — frozen target order")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            ax.text(column, row, int(matrix[row, column]), ha="center", va="center")
    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    export_figure(
        fig,
        PROJECT_ROOT / "docs/images/final_test_confusion_matrix.png",
    )
    plt.show()
else:
    print("Idempotent reuse: plots skipped to avoid redundant rendering.")

Idempotent reuse: plots skipped to avoid redundant rendering.


In [50]:
if not IDEMPOTENT_REUSE:
    aggregate_names = ["macro_f1", "balanced_accuracy", "macro_recall", "weighted_f1", "accuracy"]
    aggregate_comparison = pd.DataFrame({
        "validation": [model_selection_handoff["selected_validation_evidence"]["metrics"][name] for name in aggregate_names],
        "test": [final_evaluation.metrics[name] for name in aggregate_names],
    }, index=aggregate_names)
    aggregate_axis = aggregate_comparison.plot(kind="bar", ylim=(0.8, 1.0), title="Frozen validation vs final test")
    aggregate_figure = aggregate_axis.figure
    plt.ylabel("Score")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    export_figure(
        aggregate_figure,
        PROJECT_ROOT / "docs/images/validation_vs_test_metrics.png",
    )
    plt.show()

    recall_axis = generalization_table.set_index("class")[["validation_recall", "test_recall"]].plot(
        kind="bar", ylim=(0.75, 1.02), title="Per-class recall: validation vs final test"
    )
    recall_figure = recall_axis.figure
    plt.ylabel("Recall")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    export_figure(
        recall_figure,
        PROJECT_ROOT / "docs/images/validation_vs_test_per_class_recall.png",
    )
    plt.show()
else:
    print("Idempotent reuse: comparison plots skipped.")

Idempotent reuse: comparison plots skipped.


## 14. Multiclass Inference Contract and Lineage

The bundle requires 16 ordered numerical values, rejects missing/invalid inputs, excludes the target, and returns predicted class plus seven probabilities explicitly aligned to the target-contract class order.

In [51]:
UPSTREAM_REFERENCES = {
    "preparation": {
        "path": PREPARATION_HANDOFF_PATH.as_posix(),
        "byte_sha256": sha256_file(PROJECT_ROOT / PREPARATION_HANDOFF_PATH),
        "components": metadata_manifests["preparation_handoff"]["components"],
    },
    "model_selection": {
        "path": MODEL_SELECTION_HANDOFF_PATH.as_posix(),
        "byte_sha256": sha256_file(PROJECT_ROOT / MODEL_SELECTION_HANDOFF_PATH),
        "schema_version": model_selection_handoff["schema_version"],
    },
}
EXPECTED_INPUT_DTYPES = {
    column: feature_manifest["expected_dtypes"]["prepared"][column]
    for column in FINAL_CONTRACT.feature_columns
}
MISSING_VALUE_POLICY = {
    "strategy": "reject_missing_required_values",
    "learned_imputation_in_final_pipeline": False,
    "prepared_training_missing_value_count": 0,
    "invalid_conversion_counts": preparation_manifest["invalid_conversion_counts"],
}
print({
    "required_input_columns": list(FINAL_CONTRACT.feature_columns),
    "target_is_input": False,
    "categorical_features": list(FINAL_CONTRACT.categorical_features),
    "missing_value_behavior": "reject",
    "output_class_order": list(FINAL_CONTRACT.target_classes),
    "decision_rule": FINAL_CONTRACT.decision_rule,
})

{'required_input_columns': ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRatio', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'Roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4'], 'target_is_input': False, 'categorical_features': [], 'missing_value_behavior': 'reject', 'output_class_order': ['SEKER', 'BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SIRA'], 'decision_rule': 'argmax_class_score_or_probability'}


## 15. Atomic Final Artifact Materialization

The writer serializes the exact evaluated pipeline, validates byte and fitted-state fingerprints, proves pre/post-serialization prediction equivalence on training samples, performs technical multiclass smoke inference, stages all five files, and promotes them atomically. Equivalent complete artifacts are not overwritten.

In [52]:
if not IDEMPOTENT_REUSE:
    write_result = write_multiclass_final_model_artifacts(
        project_root=PROJECT_ROOT,
        output_directory=FINAL_MODEL_ROOT,
        pipeline=final_pipeline,
        contract=FINAL_CONTRACT,
        training_data=final_training,
        test_partition=test_partition,
        evaluation=final_evaluation,
        validation_evidence=model_selection_handoff["selected_validation_evidence"],
        fit_duration_seconds=FIT_DURATION_SECONDS,
        upstream_references=UPSTREAM_REFERENCES,
        expected_input_dtypes=EXPECTED_INPUT_DTYPES,
        missing_value_policy=MISSING_VALUE_POLICY,
        analysis_conclusions=model_selection_handoff["analysis_conclusions"],
    )
    assert write_result.idempotent is False
    print("Atomic v2 artifact set created:", list(write_result.created))
else:
    print("Idempotent reuse: no artifact was rewritten.")

Idempotent reuse: no artifact was rewritten.


## 16. Fresh-Process Reload and Training-Sample Smoke Validation

Live fitted/test objects are discarded. A new Python process validates the final handoff, bundle, every sibling hash, model SHA-256, fitted-state fingerprint, runtime contract, and training partition hash; it then loads only training rows for smoke inference.

In [53]:
if not IDEMPOTENT_REUSE:
    del final_pipeline, final_training, test_partition, final_evaluation, evaluation_guard

fresh_process_program = r'''import json
import sys
from pathlib import Path

import pandas as pd

root = Path(sys.argv[1]).resolve()
sys.path.insert(0, str(root))
from scripts.finalize_model import (
    load_and_validate_final_model_handoff,
    load_and_validate_inference_bundle,
    load_trusted_pipeline_from_bundle,
    sha256_file,
    smoke_predict_multiclass_bundle,
)

handoff_path = Path("artifacts/models/dry-bean/final-model-handoff.json")
bundle_path = Path("artifacts/models/dry-bean/inference-bundle.json")
handoff = load_and_validate_final_model_handoff(project_root=root, handoff_path=handoff_path)
bundle = load_and_validate_inference_bundle(project_root=root, bundle_path=bundle_path)
pipeline = load_trusted_pipeline_from_bundle(project_root=root, bundle=bundle)
split = json.loads((root / "artifacts/preparation/dry-bean/split-manifest.json").read_text(encoding="utf-8"))
train_path = Path(split["partition_paths"]["train"])
if sha256_file(root / train_path) != split["partition_sha256"]["train"]:
    raise RuntimeError("Training partition hash mismatch before smoke input loading.")
training_sample = pd.read_csv(root / train_path, nrows=8).loc[:, bundle["feature_columns"]]
smoke = smoke_predict_multiclass_bundle(pipeline, training_sample, bundle=bundle)
result = {
    "handoff_schema": handoff["schema_version"],
    "bundle_schema": bundle["schema_version"],
    "model_family": bundle["model_family"],
    "feature_count": len(bundle["feature_columns"]),
    "target_class_count": len(bundle["target_classes"]),
    "decision_rule": bundle["decision_rule"],
    "test_partition_evaluation_count": handoff["test_partition_evaluation_count"],
    "test_used_for_adjustment": handoff["test_partition_used_for_adjustment"],
    "model_sha256": bundle["model_artifact_sha256"],
    "model_state_fingerprint": bundle["model_state_fingerprint"],
    "smoke_rows": len(smoke),
    "smoke_probability_columns": len(smoke.iloc[0]["class_probabilities"]),
    "smoke_probability_sums_valid": all(abs(sum(values) - 1.0) <= 1e-8 for values in smoke["class_probabilities"]),
    "operational_modeling_ready": handoff["operational_modeling_ready"],
    "operational_validity": handoff["operational_validity"],
}
print(json.dumps(result, sort_keys=True))'''

completed = subprocess.run(
    [sys.executable, "-c", fresh_process_program, str(PROJECT_ROOT)],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
FRESH_PROCESS_RESULT = json.loads(completed.stdout.strip().splitlines()[-1])
assert FRESH_PROCESS_RESULT["handoff_schema"] == "final-model-handoff.v2"
assert FRESH_PROCESS_RESULT["bundle_schema"] == "inference-bundle.v2"
assert FRESH_PROCESS_RESULT["feature_count"] == 16
assert FRESH_PROCESS_RESULT["target_class_count"] == 7
assert FRESH_PROCESS_RESULT["test_partition_evaluation_count"] == 1
assert FRESH_PROCESS_RESULT["test_used_for_adjustment"] is False
assert FRESH_PROCESS_RESULT["smoke_probability_columns"] == 7
assert FRESH_PROCESS_RESULT["smoke_probability_sums_valid"] is True
print(json.dumps(FRESH_PROCESS_RESULT, indent=2, sort_keys=True))

{
  "bundle_schema": "inference-bundle.v2",
  "decision_rule": "argmax_class_score_or_probability",
  "feature_count": 16,
  "handoff_schema": "final-model-handoff.v2",
  "model_family": "HistGradientBoostingClassifier",
  "model_sha256": "b7d8aa9c5846237c58d4e9cd05ed451cad3d36931822d1b74da28c3d8db1e478",
  "model_state_fingerprint": "0f301110a38db2e60d3c41432c69cc7458e301fd8865858f3ac9b853b87909aa",
  "operational_modeling_ready": false,
  "operational_validity": "unconfirmed",
  "smoke_probability_columns": 7,
  "smoke_probability_sums_valid": true,
  "smoke_rows": 8,
  "target_class_count": 7,
  "test_partition_evaluation_count": 1,
  "test_used_for_adjustment": false
}


## 17. Final Handoff and Readiness

The v2 handoff is sufficient for a future fresh-process Notebook 05 loader and seven-class inference demonstration, but Notebook 05 is not implemented here. Production readiness remains blocked by absent operational data contracts, monitoring, SLOs, retraining, and deployment validation. ShapeFactor2 remains frozen in the model while its source provenance remains unresolved.

In [54]:
final_handoff = json.loads(
    (PROJECT_ROOT / FINAL_MODEL_ROOT / "final-model-handoff.json").read_text(encoding="utf-8")
)
final_manifest = json.loads(
    (PROJECT_ROOT / FINAL_MODEL_ROOT / "final-model-manifest.json").read_text(encoding="utf-8")
)
final_test_evidence = json.loads(
    (PROJECT_ROOT / FINAL_MODEL_ROOT / "final-test-evidence.json").read_text(encoding="utf-8")
)
inference_bundle = json.loads(
    (PROJECT_ROOT / FINAL_MODEL_ROOT / "inference-bundle.json").read_text(encoding="utf-8")
)

READINESS_FIELDS = [
    "preparation_handoff_validated",
    "model_selection_handoff_validated",
    "frozen_finalization_contract_validated",
    "final_training_completed",
    "final_model_trained",
    "test_partition_opened_after_final_fit",
    "final_test_evaluation_completed",
    "test_partition_evaluation_count",
    "test_partition_used_for_adjustment",
    "model_artifact_materialized",
    "model_bundle_materialized",
    "serialization_reload_validated",
    "inference_smoke_test_completed",
    "educational_final_model_completed",
    "educational_inference_demo_ready",
    "final_model_handoff_ready",
    "operational_modeling_ready",
    "operational_validity",
]
readiness = {field: final_handoff[field] for field in READINESS_FIELDS}
assert final_handoff["no_model_selection_decision_changed_after_test"] is True
assert final_handoff["operational_modeling_ready"] is False
assert final_handoff["operational_validity"] == "unconfirmed"
assert final_handoff["analysis_conclusions"]["shape_factor_2"]["provenance_status"] == "unresolved"

print(json.dumps(readiness, indent=2, sort_keys=True))
print("Final test metrics:", json.dumps(final_test_evidence["metrics"], indent=2, sort_keys=True))
print("Model artifact:", inference_bundle["model_artifact_path"])
print("Model SHA-256:", inference_bundle["model_artifact_sha256"])
print("Model-state fingerprint:", inference_bundle["model_state_fingerprint"])
print("Notebook 05 readiness is educational only; no inference demo was implemented here.")

{
  "educational_final_model_completed": true,
  "educational_inference_demo_ready": true,
  "final_model_handoff_ready": true,
  "final_model_trained": true,
  "final_test_evaluation_completed": true,
  "final_training_completed": true,
  "frozen_finalization_contract_validated": true,
  "inference_smoke_test_completed": true,
  "model_artifact_materialized": true,
  "model_bundle_materialized": true,
  "model_selection_handoff_validated": true,
  "operational_modeling_ready": false,
  "operational_validity": "unconfirmed",
  "preparation_handoff_validated": true,
  "serialization_reload_validated": true,
  "test_partition_evaluation_count": 1,
  "test_partition_opened_after_final_fit": true,
  "test_partition_used_for_adjustment": false
}
Final test metrics: {
  "accuracy": 0.9324191968658179,
  "balanced_accuracy": 0.9398971908347276,
  "log_loss": 0.1810239003137349,
  "macro_f1": 0.9418353636024855,
  "macro_recall": 0.9398971908347276,
  "minimum_per_class_recall": 0.868686868686